# Libreias 

In [18]:
import requests
import pandas as pd
import numpy as np
import os 
import re 

pd.set_option('display.max_columns', 50)

BASE_URL = "https://akabab.github.io/superhero-api/api/all.json"

# Consumir la api crudo 

In [19]:
resp = requests.get(BASE_URL)
resp.raise_for_status() # Lanza un error si la solicitud 

heroes = resp.json()

len(heroes)

563

In [20]:
df_raw = pd.json_normalize(heroes)
df_raw.head()

,id,name,slug,powerstats.intelligence,powerstats.strength,powerstats.speed,powerstats.durability,powerstats.power,powerstats.combat,appearance.gender,appearance.race,appearance.height,appearance.weight,appearance.eyeColor,appearance.hairColor,biography.fullName,biography.alterEgos,biography.aliases,biography.placeOfBirth,biography.firstAppearance,biography.publisher,biography.alignment,work.occupation,work.base,connections.groupAffiliation,connections.relatives,images.xs,images.sm,images.md,images.lg
0,1,A-Bomb,1-a-bomb,38,100,17,80,24,64,Male,Human,"[6'8, 203 cm]","[980 lb, 441 kg]",Yellow,No Hair,Richard Milhouse Jones,No alter egos found.,[Rick Jones],"Scarsdale, Arizona","Hulk Vol 2 #2 (April, 2008) (as A-Bomb)",Marvel Comics,good,"Musician, adventurer, author; formerly talk sh...",-,"Hulk Family; Excelsior (sponsor), Avengers (ho...",Marlo Chandler-Jones (wife); Polly (aunt); Mrs...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...
1,2,Abe Sapien,2-abe-sapien,88,28,35,65,100,85,Male,Icthyo Sapien,"[6'3, 191 cm]","[145 lb, 65 kg]",Blue,No Hair,Abraham Sapien,No alter egos found.,"[Langdon Everett Caul, Abraham Sapien, Langdon...",-,Hellboy: Seed of Destruction (1993),Dark Horse Comics,good,Paranormal Investigator,-,Bureau for Paranormal Research and Defense,"Edith Howard (wife, deceased)",https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...
2,3,Abin Sur,3-abin-sur,50,90,53,64,99,65,Male,Ungaran,"[6'1, 185 cm]","[200 lb, 90 kg]",Blue,No Hair,,No alter egos found.,[Lagzia],Ungara,"Showcase #22 (October, 1959)",DC Comics,good,"Green Lantern, former history professor",Oa,"Green Lantern Corps, Black Lantern Corps","Amon Sur (son), Arin Sur (sister), Thaal Sines...",https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...
3,4,Abomination,4-abomination,63,80,53,90,62,95,Male,Human / Radiation,"[6'8, 203 cm]","[980 lb, 441 kg]",Green,No Hair,Emil Blonsky,No alter egos found.,"[Agent R-7, Ravager of Worlds]","Zagreb, Yugoslavia",Tales to Astonish #90,Marvel Comics,bad,Ex-Spy,Mobile,former member of the crew of the Andromeda Sta...,"Nadia Dornova Blonsky (wife, separated)",https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...
4,5,Abraxas,5-abraxas,88,63,83,100,100,55,Male,Cosmic Entity,"[-, 0 cm]","[- lb, 0 kg]",Blue,Black,Abraxas,No alter egos found.,[-],Within Eternity,Fantastic Four Annual #2001,Marvel Comics,bad,Dimensional destroyer,-,Cosmic Beings,"Eternity (""Father"")",https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...,https://cdn.jsdelivr.net/gh/akabab/superhero-a...


In [22]:
df_raw["appearance.weight"].head(20)

0     [980 lb, 441 kg]
1      [145 lb, 65 kg]
2      [200 lb, 90 kg]
3     [980 lb, 441 kg]
4         [- lb, 0 kg]
5     [270 lb, 122 kg]
6         [- lb, 0 kg]
7      [195 lb, 88 kg]
8      [181 lb, 81 kg]
9     [230 lb, 104 kg]
10    [240 lb, 108 kg]
11     [200 lb, 90 kg]
12     [201 lb, 90 kg]
13        [- lb, 0 kg]
14     [160 lb, 72 kg]
15    [375 lb, 169 kg]
16    [385 lb, 173 kg]
17        [- lb, 0 kg]
18     [150 lb, 68 kg]
19     [126 lb, 57 kg]
Name: appearance.weight, dtype: object

# Limpiar 

In [28]:
def parse_height(height_list):
    """
    Convierte height a cm usando la regla: 
    1) Usar cm si es valido y si no 'O cm'
    2 ) Si cm es invalido , intnentar convertir de ft a cm
    """

    if not height_list or len(height_list) == 0:
        return pd.NA
    
    h1 = str(height_list[0])
    h2 = str(height_list[1]) if len(height_list) > 1 else '0 cm'

    # Intentar usar cm 

    if h2 and "cm" in h2 and h2 not in ['0 cm', "- ", "null"]:
        try: 
            return float(h2.replace("cm", "").strip()) 
        except: 
            pass
    # Intentar convertir de ft a cm
    pattern  = r"([\d\.]+)\s*ft"
    match = re.search(pattern, h1)
    if match:
        feet = int(match.group(1))
        inches = int(match.group(2)) 
        return round((feet * 30.48 + inches * 2.54), 2)
    return pd.NA

In [26]:
def parse_weight(weight_list): 
    """
    Convertir weight a kg usando la regla:
    1)  Usar kg validos 
    2 ) Converitr libaras si kg no es valido
    """
    if not weight_list or len(weight_list) == 0:
        return pd.NA
    w1 = str(weight_list[0])
    w2 = str(weight_list[1]) if len(weight_list) > 1 else None 

    # Intentar usar kg
    if w2 and "kg" in w2 and w2 not in ['0 kg', "- ", "null"]:
        try: 
            return float(w2.replace("kg", "").strip()) 
        except: 
            pass
    # Intentar convertir de libras a kg

    if "lb" in w1 and w1 not in ['- lb', "-lb"]: 
        try: 
            lbs =  float(w1.replace("lb", "").replace("-", "").strip())
            return round(lbs * 0.453592, 2)
        except:
            pass
    return pd.NA

In [29]:
sample_heights = df_raw["appearance.height"].head(10).tolist()
sample_weights = df_raw["appearance.weight"].head(10).tolist()

for h in sample_heights:
    print(h, "->", parse_height(h))
print("\n------------------\n")
    
for w in sample_weights:
    print(w, "->", parse_weight(w))

["6'8", '203 cm'] -> 203.0
["6'3", '191 cm'] -> 191.0
["6'1", '185 cm'] -> 185.0
["6'8", '203 cm'] -> 203.0
['-', '0 cm'] -> <NA>
["6'4", '193 cm'] -> 193.0
['-', '0 cm'] -> <NA>
["6'1", '185 cm'] -> 185.0
["5'10", '178 cm'] -> 178.0
["6'3", '191 cm'] -> 191.0

------------------

['980 lb', '441 kg'] -> 441.0
['145 lb', '65 kg'] -> 65.0
['200 lb', '90 kg'] -> 90.0
['980 lb', '441 kg'] -> 441.0
['- lb', '0 kg'] -> <NA>
['270 lb', '122 kg'] -> 122.0
['- lb', '0 kg'] -> <NA>
['195 lb', '88 kg'] -> 88.0
['181 lb', '81 kg'] -> 81.0
['230 lb', '104 kg'] -> 104.0


In [30]:
df_clean = pd.DataFrame({
    "intelligence": df_raw["powerstats.intelligence"],
    "strength": df_raw["powerstats.strength"],
    "speed" : df_raw["powerstats.speed"],
    "durability": df_raw["powerstats.durability"],
    "combat": df_raw["powerstats.combat"],
    "power": df_raw["powerstats.power"],
    "height_cm": df_raw["appearance.height"].apply(parse_height),
    "weight_kg": df_raw["appearance.weight"].apply(parse_weight), 
})
df_clean = df_clean.apply(pd.to_numeric, errors='coerce')
df_clean.head()

,intelligence,strength,speed,durability,combat,power,height_cm,weight_kg
0,38,100,17,80,64,24,203.0,441.0
1,88,28,35,65,85,100,191.0,65.0
2,50,90,53,64,65,99,185.0,90.0
3,63,80,53,90,95,62,203.0,441.0
4,88,63,83,100,55,100,NaN,NaN


In [35]:
# Drop na 
df_clean = df_clean.dropna().reset_index(drop=True)
df_clean.head()

,intelligence,strength,speed,durability,combat,power,height_cm,weight_kg
0,38,100,17,80,64,24,203.0,441.0
1,88,28,35,65,85,100,191.0,65.0
2,50,90,53,64,65,99,185.0,90.0
3,63,80,53,90,95,62,203.0,441.0
4,38,80,25,100,64,98,193.0,122.0


# Ruido para llegar a 600

In [37]:
TARGET_SIZE = 600

len_original = len(df_clean)
print("Original size:", len_original)

Original size: 428


In [40]:
if len_original < TARGET_SIZE:
    needed = TARGET_SIZE - len_original
    print(f"Faltan {needed} registros para llegar a ", TARGET_SIZE)

    df_extra = df_clean.sample(n=needed, replace=True, random_state=42).copy()
    # Ruido suave de 0.2 std en todas las columnas numericas 

    for col in df_extra.columns:
        if pd.api.types.is_numeric_dtype(df_extra[col]):
            noise = np.random.normal(0, 0.2, size=len(df_extra))
            df_extra[col] = df_extra[col] + noise
    # Redondear 
    df_extra = df_extra.round(1)
    # Combianr el original con el extra que creamos 
    df_final = pd.concat([df_clean, df_extra], ignore_index=True)
else : 
    print (" Hay mas de 600 filas , se seleccionaron 600 al azar")
    df_final = df_clean.copy()

# Veamos que si sean 600
df_final = df_final.sample(n=TARGET_SIZE, random_state=42).reset_index(drop=True)
df_final.shape

Faltan 172 registros para llegar a  600


(600, 8)